# Chapter 11.1: Development Environment Setup - Build from Source

This notebook covers setting up a complete development environment for contributing to vLLM and SGLang.
We will walk through building from source, configuring IDEs, and setting up pre-commit hooks.

**Learning Objectives:**
- Build vLLM and SGLang from source
- Configure CUDA toolkit and Python environments
- Set up IDE with proper extensions
- Configure pre-commit hooks
- Troubleshoot common build issues

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part11/chapter_11.1_dev_setup.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part11/chapter_11.1_dev_setup.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_development_workflow():
    """Development workflow: Clone -> Branch -> Build -> Changes -> Test -> Lint -> PR -> Review -> Merge."""
    fig, ax = plt.subplots(figsize=(18, 6))
    ax.set_xlim(0, 20); ax.set_ylim(0, 5); ax.axis('off')
    ax.set_title('Development Workflow for Contributing to vLLM / SGLang',
                 fontsize=15, fontweight='bold', pad=15)

    steps = [
        ('Clone\nRepo', '#4A90D9'),
        ('Create\nBranch', '#4A90D9'),
        ('Build from\nSource', '#27AE60'),
        ('Make\nChanges', '#F39C12'),
        ('Run\nTests', '#E74C3C'),
        ('Lint &\nFormat', '#8E44AD'),
        ('Create\nPR', '#4A90D9'),
        ('Code\nReview', '#27AE60'),
        ('Merge', '#27AE60'),
    ]

    box_w, box_h = 1.7, 2.0
    y_center = 2.5
    x_start = 0.5
    spacing = 2.1

    for i, (label, color) in enumerate(steps):
        x = x_start + i * spacing
        rect = FancyBboxPatch((x, y_center - box_h/2), box_w, box_h,
                              boxstyle='round,pad=0.1', facecolor=color,
                              edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(rect)
        ax.text(x + box_w/2, y_center, label, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
        # Step number
        circle = plt.Circle((x + box_w/2, y_center + box_h/2 + 0.25), 0.2,
                            facecolor=color, edgecolor='white', linewidth=1.5)
        ax.add_patch(circle)
        ax.text(x + box_w/2, y_center + box_h/2 + 0.25, str(i+1),
                ha='center', va='center', fontsize=7, fontweight='bold', color='white')

        if i < len(steps) - 1:
            ax.annotate('', xy=(x + box_w + 0.05, y_center),
                        xytext=(x + box_w + spacing - box_w - 0.05, y_center),
                        arrowprops=dict(arrowstyle='<-', color='#333', lw=1.8))

    # Feedback loop from Review back to Changes
    ax.annotate('', xy=(x_start + 3 * spacing + box_w/2, y_center - box_h/2 - 0.3),
                xytext=(x_start + 7 * spacing + box_w/2, y_center - box_h/2 - 0.3),
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5,
                                connectionstyle='arc3,rad=-0.3', linestyle='dashed'))
    ax.text(10, 0.3, 'revise if needed', ha='center', fontsize=7, color='#E74C3C', style='italic')

    plt.tight_layout()
    plt.savefig('development_workflow.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_development_workflow()

## 1. Prerequisites Check

Before building from source, we need to verify that all prerequisites are installed.

In [ ]:
# Demo: Check system prerequisites
import subprocess
import sys
import os
import shutil

def check_command(cmd, version_flag="--version"):
    """Check if a command is available and print its version."""
    try:
        result = subprocess.run(
            [cmd, version_flag],
            capture_output=True, text=True, timeout=10
        )
        output = result.stdout.strip() or result.stderr.strip()
        first_line = output.split('\n')[0] if output else 'unknown'
        print(f"[OK] {cmd}: {first_line}")
        return True
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print(f"[MISSING] {cmd} not found")
        return False

print("=" * 60)
print("Development Environment Prerequisites Check")
print("=" * 60)

# Python
print(f"\n--- Python ---")
print(f"[OK] Python: {sys.version}")
print(f"[OK] Python path: {sys.executable}")

# Build tools
print(f"\n--- Build Tools ---")
check_command("gcc")
check_command("g++")
check_command("cmake")
check_command("make")
check_command("git")

# CUDA
print(f"\n--- CUDA ---")
check_command("nvcc")
check_command("nvidia-smi")

In [ ]:
# Demo: Check CUDA toolkit and driver compatibility
import subprocess

def get_cuda_info():
    """Gather detailed CUDA information."""
    info = {}
    
    # Driver version from nvidia-smi
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version,gpu_name,memory.total",
             "--format=csv,noheader"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for i, line in enumerate(lines):
                parts = [p.strip() for p in line.split(',')]
                info[f'gpu_{i}'] = {
                    'driver': parts[0],
                    'name': parts[1],
                    'memory': parts[2]
                }
    except FileNotFoundError:
        info['error'] = 'nvidia-smi not found'
    
    # CUDA toolkit version
    try:
        result = subprocess.run(
            ["nvcc", "--version"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            for line in result.stdout.split('\n'):
                if 'release' in line:
                    info['cuda_toolkit'] = line.strip()
    except FileNotFoundError:
        info['cuda_toolkit'] = 'nvcc not found'
    
    return info

cuda_info = get_cuda_info()
print("CUDA Environment Information:")
print("-" * 40)
for key, value in cuda_info.items():
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

# CUDA/Driver compatibility matrix
print("\n" + "=" * 60)
print("CUDA Toolkit / Driver Compatibility (minimum driver versions):")
print("-" * 60)
compat = [
    ("CUDA 12.4", ">= 550.54.14", "Recommended for vLLM 0.6+"),
    ("CUDA 12.1", ">= 530.30.02", "Minimum for vLLM 0.5+"),
    ("CUDA 11.8", ">= 520.61.05", "Legacy support"),
]
for cuda, driver, note in compat:
    print(f"  {cuda:12s} | Driver {driver:16s} | {note}")

## 2. Python Environment Management

We strongly recommend using **conda** or **venv** to isolate the development environment.

In [ ]:
# Demo: Create and manage Python environments for vLLM/SGLang development

# Option A: Using conda (recommended for CUDA management)
conda_commands = """
# Create a new conda environment for vLLM development
conda create -n vllm-dev python=3.11 -y
conda activate vllm-dev

# Install CUDA toolkit via conda (ensures compatibility)
conda install -c nvidia cuda-toolkit=12.4 -y

# Verify CUDA is from conda
which nvcc
nvcc --version
"""

# Option B: Using venv
venv_commands = """
# Create a virtual environment
python3.11 -m venv ~/vllm-dev-env
source ~/vllm-dev-env/bin/activate

# Upgrade pip
pip install --upgrade pip setuptools wheel
"""

print("Option A: Conda Environment Setup")
print("=" * 50)
print(conda_commands)

print("\nOption B: venv Environment Setup")
print("=" * 50)
print(venv_commands)

# Show current environment info
print("\nCurrent Python Environment:")
print(f"  Executable: {sys.executable}")
print(f"  Version: {sys.version}")
print(f"  Prefix: {sys.prefix}")
print(f"  Virtual env: {hasattr(sys, 'real_prefix') or sys.base_prefix != sys.prefix}")

## 3. Building vLLM from Source

Let's walk through the complete process of building vLLM from source.

In [ ]:
# Demo: Clone and build vLLM from source - step by step

build_script = '''
#!/bin/bash
set -e

echo "=== Step 1: Clone the vLLM repository ==="
git clone https://github.com/vllm-project/vllm.git
cd vllm

echo "=== Step 2: Check out the latest release or main branch ==="
# For development, use main:
git checkout main
# Or a specific version:
# git checkout v0.6.0

echo "=== Step 3: Install build dependencies ==="
pip install -r requirements-build.txt

echo "=== Step 4: Install runtime dependencies ==="
pip install -r requirements-common.txt
pip install -r requirements-cuda.txt  # For CUDA builds

echo "=== Step 5: Build and install in development mode ==="
# Development mode (-e) allows code changes without reinstall
pip install -e . --no-build-isolation

echo "=== Step 6: Verify installation ==="
python -c "import vllm; print(f'vLLM version: {vllm.__version__}')"
'''

print("vLLM Build-from-Source Script")
print("=" * 60)
print(build_script)

# Environment variables that control the build
print("\nImportant Environment Variables for the Build:")
print("-" * 60)
env_vars = [
    ("VLLM_TARGET_DEVICE", "cuda", "Target device (cuda, rocm, cpu, etc.)"),
    ("CUDA_HOME", "/usr/local/cuda", "Path to CUDA toolkit"),
    ("MAX_JOBS", "$(nproc)", "Max parallel compilation jobs"),
    ("VLLM_INSTALL_PUNICA_KERNELS", "0", "Install LoRA kernels"),
    ("CMAKE_BUILD_TYPE", "RelWithDebInfo", "Build type for debugging"),
]
for var, default, desc in env_vars:
    current = os.environ.get(var, "not set")
    print(f"  {var}")
    print(f"    Default: {default}")
    print(f"    Current: {current}")
    print(f"    Purpose: {desc}")

In [ ]:
# Demo: Understanding the vLLM project structure after cloning

vllm_structure = {
    "vllm/": {
        "description": "Main Python package",
        "subdirs": {
            "attention/": "Attention backends (flash-attn, paged, etc.)",
            "core/": "Core scheduler and block manager",
            "engine/": "LLM engine (sync and async)",
            "entrypoints/": "API server, CLI, OpenAI-compatible endpoints",
            "executor/": "Execution backends (GPU, Ray, etc.)",
            "model_executor/": "Model loading and execution",
            "worker/": "GPU worker processes",
            "lora/": "LoRA adapter support",
            "spec_decode/": "Speculative decoding",
            "transformers_utils/": "HuggingFace integration utilities",
        }
    },
    "csrc/": {
        "description": "C++/CUDA source code",
        "subdirs": {
            "attention/": "Custom attention CUDA kernels",
            "quantization/": "Quantization kernels (GPTQ, AWQ, etc.)",
            "cache_kernels.cu": "KV cache operations",
            "pos_encoding_kernels.cu": "Rotary positional encoding",
        }
    },
    "tests/": {
        "description": "Test suite",
        "subdirs": {
            "models/": "Model correctness tests",
            "kernels/": "CUDA kernel tests",
            "core/": "Scheduler and block manager tests",
            "entrypoints/": "API server tests",
        }
    }
}

def print_structure(structure, indent=0):
    for name, info in structure.items():
        prefix = "  " * indent
        if isinstance(info, dict) and 'description' in info:
            print(f"{prefix}{name}  # {info['description']}")
            if 'subdirs' in info:
                for subname, subdesc in info['subdirs'].items():
                    print(f"{prefix}  {subname}  # {subdesc}")
        else:
            print(f"{prefix}{name}  # {info}")

print("vLLM Project Structure")
print("=" * 60)
print_structure(vllm_structure)

## 4. Building SGLang from Source

In [ ]:
# Demo: Clone and build SGLang from source

sglang_build_script = '''
#!/bin/bash
set -e

echo "=== Step 1: Clone SGLang ==="
git clone https://github.com/sgl-project/sglang.git
cd sglang

echo "=== Step 2: Install in development mode ==="
# SGLang has a simpler build process
pip install -e "python[all]" --no-build-isolation

# Or install specific components:
# pip install -e "python[srt]"   # SGLang Runtime only
# pip install -e "python[openai]" # OpenAI backend

echo "=== Step 3: Install development dependencies ==="
pip install -r requirements-dev.txt

echo "=== Step 4: Build custom CUDA kernels (if needed) ==="
# SGLang may have custom kernels that need separate compilation
cd python/sglang/srt/layers/quantization
python setup.py build_ext --inplace

echo "=== Step 5: Verify installation ==="
python -c "import sglang; print('SGLang installed successfully')"
'''

print("SGLang Build-from-Source Script")
print("=" * 60)
print(sglang_build_script)

# SGLang project structure
sglang_structure = """
SGLang Project Structure:
========================
sglang/
  python/
    sglang/
      srt/               # SGLang Runtime (the inference engine)
        layers/           # Model layers and kernels
        managers/         # Memory and request managers  
        model_executor/   # Model execution logic
        server.py         # HTTP server
      lang/              # SGLang language frontend
        ir.py            # Intermediate representation
        interpreter.py   # Program interpreter
  test/                  # Test suite
    srt/                 # Runtime tests
    lang/                # Language tests
  benchmark/             # Benchmark scripts
  docs/                  # Documentation
"""
print(sglang_structure)

## 5. IDE Setup

In [ ]:
# Demo: Generate VSCode settings for vLLM/SGLang development
import json

vscode_settings = {
    "python.defaultInterpreterPath": "${workspaceFolder}/.venv/bin/python",
    "python.analysis.typeCheckingMode": "basic",
    "python.analysis.autoImportCompletions": True,
    "python.analysis.extraPaths": [
        "${workspaceFolder}",
        "${workspaceFolder}/csrc"
    ],
    "python.testing.pytestEnabled": True,
    "python.testing.pytestArgs": [
        "tests",
        "-v",
        "--tb=short"
    ],
    "editor.formatOnSave": True,
    "editor.rulers": [80, 120],
    "[python]": {
        "editor.defaultFormatter": "ms-python.black-formatter",
        "editor.codeActionsOnSave": {
            "source.organizeImports": "explicit"
        }
    },
    "files.associations": {
        "*.cu": "cuda-cpp",
        "*.cuh": "cuda-cpp"
    },
    "C_Cpp.default.includePath": [
        "${workspaceFolder}/csrc",
        "/usr/local/cuda/include",
        "${workspaceFolder}/.venv/lib/python3.11/site-packages/torch/include"
    ]
}

print("VSCode settings.json for vLLM development:")
print("=" * 60)
print(json.dumps(vscode_settings, indent=4))

# Recommended extensions
extensions = {
    "extensions.json": {
        "recommendations": [
            "ms-python.python",
            "ms-python.vscode-pylance",
            "ms-python.black-formatter",
            "ms-python.isort",
            "ms-vscode.cpptools",
            "nvidia.nsight-vscode-edition",
            "ms-python.debugpy",
            "eamodio.gitlens",
            "github.copilot"
        ]
    }
}
print("\nRecommended VSCode extensions:")
print("-" * 40)
for ext in extensions["extensions.json"]["recommendations"]:
    print(f"  - {ext}")

In [ ]:
# Demo: Generate launch.json for debugging vLLM
import json

launch_config = {
    "version": "0.2.0",
    "configurations": [
        {
            "name": "Debug vLLM API Server",
            "type": "debugpy",
            "request": "launch",
            "module": "vllm.entrypoints.openai.api_server",
            "args": [
                "--model", "facebook/opt-125m",
                "--port", "8000",
                "--max-model-len", "512"
            ],
            "env": {
                "CUDA_VISIBLE_DEVICES": "0"
            },
            "justMyCode": False
        },
        {
            "name": "Debug vLLM Offline Inference",
            "type": "debugpy",
            "request": "launch",
            "program": "${workspaceFolder}/examples/offline_inference.py",
            "env": {
                "CUDA_VISIBLE_DEVICES": "0"
            },
            "justMyCode": False
        },
        {
            "name": "Debug vLLM Tests",
            "type": "debugpy",
            "request": "launch",
            "module": "pytest",
            "args": [
                "tests/core/test_scheduler.py",
                "-v", "-x", "--tb=short"
            ],
            "justMyCode": False
        },
        {
            "name": "Attach to Running vLLM Process",
            "type": "debugpy",
            "request": "attach",
            "connect": {
                "host": "localhost",
                "port": 5678
            },
            "justMyCode": False
        }
    ]
}

print("VSCode launch.json for debugging vLLM:")
print("=" * 60)
print(json.dumps(launch_config, indent=4))

## 6. Pre-commit Hooks Setup

In [ ]:
# Demo: Set up pre-commit hooks for vLLM development
import yaml

precommit_config = {
    "repos": [
        {
            "repo": "https://github.com/pre-commit/pre-commit-hooks",
            "rev": "v4.5.0",
            "hooks": [
                {"id": "trailing-whitespace"},
                {"id": "end-of-file-fixer"},
                {"id": "check-yaml"},
                {"id": "check-added-large-files", "args": ["--maxkb=1000"]},
                {"id": "check-merge-conflict"}
            ]
        },
        {
            "repo": "https://github.com/psf/black",
            "rev": "24.3.0",
            "hooks": [
                {"id": "black", "language_version": "python3"}
            ]
        },
        {
            "repo": "https://github.com/pycqa/isort",
            "rev": "5.13.2",
            "hooks": [
                {"id": "isort", "args": ["--profile", "black"]}
            ]
        },
        {
            "repo": "https://github.com/pycqa/flake8",
            "rev": "7.0.0",
            "hooks": [
                {"id": "flake8", "args": ["--max-line-length=120", "--ignore=E203,W503"]}
            ]
        },
        {
            "repo": "https://github.com/pre-commit/mirrors-mypy",
            "rev": "v1.8.0",
            "hooks": [
                {"id": "mypy", "args": ["--ignore-missing-imports"]}
            ]
        }
    ]
}

print("Pre-commit configuration (.pre-commit-config.yaml):")
print("=" * 60)
print(yaml.dump(precommit_config, default_flow_style=False))

# Installation commands
install_cmds = """
# Install pre-commit
pip install pre-commit

# Install the hooks into your local repo
cd /path/to/vllm
pre-commit install

# Run on all files (first time)
pre-commit run --all-files

# Run on staged files only
pre-commit run

# Update hooks to latest versions
pre-commit autoupdate
"""
print("\nInstallation Commands:")
print(install_cmds)

## 7. Running the Development Server

In [ ]:
# Demo: Running vLLM in development mode with hot reload

dev_server_script = '''
#!/bin/bash
# dev_server.sh - Start vLLM dev server with debugging enabled

export CUDA_VISIBLE_DEVICES=0
export VLLM_LOGGING_LEVEL=DEBUG
export VLLM_TRACE_FUNCTION=1  # Enable function tracing

# Option 1: Run with a small model for fast iteration
python -m vllm.entrypoints.openai.api_server \\
    --model facebook/opt-125m \\
    --port 8000 \\
    --max-model-len 512 \\
    --enforce-eager  # Disable CUDA graphs for debugging

# Option 2: Run with debugpy for remote debugging
# python -m debugpy --listen 5678 --wait-for-client \\
#     -m vllm.entrypoints.openai.api_server \\
#     --model facebook/opt-125m \\
#     --port 8000
'''

sglang_dev_script = '''
#!/bin/bash
# dev_sglang.sh - Start SGLang dev server

export CUDA_VISIBLE_DEVICES=0

python -m sglang.launch_server \\
    --model-path facebook/opt-125m \\
    --port 30000 \\
    --log-level debug
'''

print("vLLM Development Server Script:")
print("=" * 60)
print(dev_server_script)

print("\nSGLang Development Server Script:")
print("=" * 60)
print(sglang_dev_script)

# Quick test to verify the server
test_script = '''
# Test vLLM server with curl
curl -s http://localhost:8000/v1/completions \\
  -H "Content-Type: application/json" \\
  -d '{
    "model": "facebook/opt-125m",
    "prompt": "Hello, world!",
    "max_tokens": 10
  }' | python -m json.tool

# Test SGLang server
curl -s http://localhost:30000/v1/completions \\
  -H "Content-Type: application/json" \\
  -d '{
    "model": "facebook/opt-125m",
    "prompt": "Hello, world!",
    "max_tokens": 10
  }' | python -m json.tool
'''
print("\nTest Commands:")
print(test_script)

## 8. Troubleshooting Common Build Issues

In [ ]:
# Demo: Common build troubleshooting guide with diagnostic scripts

class BuildDiagnostic:
    """Diagnostic helper for common build issues."""
    
    def __init__(self):
        self.issues = []
    
    def check_cuda_home(self):
        cuda_home = os.environ.get('CUDA_HOME') or os.environ.get('CUDA_PATH')
        if not cuda_home:
            # Try common locations
            for path in ['/usr/local/cuda', '/usr/lib/cuda', '/opt/cuda']:
                if os.path.exists(path):
                    cuda_home = path
                    break
        
        if cuda_home and os.path.exists(cuda_home):
            print(f"[OK] CUDA_HOME: {cuda_home}")
            # Check for nvcc
            nvcc = os.path.join(cuda_home, 'bin', 'nvcc')
            if os.path.exists(nvcc):
                print(f"[OK] nvcc found: {nvcc}")
            else:
                self.issues.append(f"nvcc not found at {nvcc}")
                print(f"[WARN] nvcc not found at {nvcc}")
        else:
            self.issues.append("CUDA_HOME not set or not found")
            print("[WARN] CUDA_HOME not set")
            print("  Fix: export CUDA_HOME=/usr/local/cuda")
    
    def check_torch_cuda(self):
        try:
            import torch
            print(f"[OK] PyTorch: {torch.__version__}")
            print(f"  CUDA available: {torch.cuda.is_available()}")
            if torch.cuda.is_available():
                print(f"  CUDA version: {torch.version.cuda}")
                print(f"  GPU count: {torch.cuda.device_count()}")
                for i in range(torch.cuda.device_count()):
                    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
            else:
                self.issues.append("PyTorch CUDA not available")
        except ImportError:
            self.issues.append("PyTorch not installed")
            print("[MISSING] PyTorch not installed")
    
    def check_gcc_version(self):
        try:
            result = subprocess.run(
                ['gcc', '--version'], capture_output=True, text=True
            )
            version_line = result.stdout.split('\n')[0]
            print(f"[OK] GCC: {version_line}")
            # Check if version >= 7 (required for C++17)
        except FileNotFoundError:
            self.issues.append("GCC not found")
            print("[MISSING] GCC not found")
    
    def print_summary(self):
        print("\n" + "=" * 60)
        if self.issues:
            print(f"Found {len(self.issues)} issue(s):")
            for i, issue in enumerate(self.issues, 1):
                print(f"  {i}. {issue}")
        else:
            print("All checks passed! Ready to build.")

diag = BuildDiagnostic()
print("Running Build Diagnostics...")
print("=" * 60)
diag.check_cuda_home()
print()
diag.check_torch_cuda()
print()
diag.check_gcc_version()
diag.print_summary()

In [ ]:
# Demo: Common build errors and their solutions

common_errors = [
    {
        "error": "error: command 'nvcc' failed: No such file or directory",
        "cause": "CUDA toolkit not installed or not in PATH",
        "solution": [
            "export CUDA_HOME=/usr/local/cuda",
            "export PATH=$CUDA_HOME/bin:$PATH",
            "export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH"
        ]
    },
    {
        "error": "error: #error -- unsupported GNU version! gcc versions later than 12 are not supported",
        "cause": "GCC version too new for CUDA",
        "solution": [
            "# Install compatible GCC version",
            "sudo apt install gcc-11 g++-11",
            "export CC=gcc-11",
            "export CXX=g++-11"
        ]
    },
    {
        "error": "ModuleNotFoundError: No module named 'torch'",
        "cause": "PyTorch not installed before building vLLM",
        "solution": [
            "pip install torch==2.4.0 --index-url https://download.pytorch.org/whl/cu124"
        ]
    },
    {
        "error": "fatal error: Python.h: No such file or directory",
        "cause": "Python development headers not installed",
        "solution": [
            "sudo apt install python3.11-dev",
            "# or with conda: conda install python=3.11"
        ]
    },
    {
        "error": "CMake Error: Could not find a package configuration file for 'Torch'",
        "cause": "PyTorch not found by CMake",
        "solution": [
            "pip install torch  # ensure torch is installed in the active env",
            "# Verify: python -c 'import torch; print(torch.utils.cmake_prefix_path)'"
        ]
    },
    {
        "error": "error: out of memory (build process killed)",
        "cause": "Too many parallel compilation jobs",
        "solution": [
            "export MAX_JOBS=2  # Reduce parallel jobs",
            "pip install -e . --no-build-isolation"
        ]
    }
]

print("Common Build Errors and Solutions")
print("=" * 70)
for i, err in enumerate(common_errors, 1):
    print(f"\n--- Error {i} ---")
    print(f"Error:    {err['error']}")
    print(f"Cause:    {err['cause']}")
    print(f"Solution:")
    for cmd in err['solution']:
        print(f"    {cmd}")

## 9. Development Workflow Helper Scripts

In [ ]:
# Demo: Create development workflow helper scripts

# Script to rebuild only CUDA kernels (much faster than full rebuild)
rebuild_kernels = '''
#!/bin/bash
# rebuild_kernels.sh - Rebuild only the CUDA kernels
# Useful when modifying csrc/ files

set -e
cd "$(git rev-parse --show-toplevel)"  # Go to repo root

echo "Rebuilding CUDA kernels..."
python setup.py build_ext --inplace
echo "Done! Kernels rebuilt."
'''

# Script for quick validation before pushing
pre_push_check = '''
#!/bin/bash
# pre_push_check.sh - Quick validation before pushing
set -e
cd "$(git rev-parse --show-toplevel)"

echo "=== Step 1: Formatting ==="
python -m black --check vllm/ tests/
python -m isort --check-only --profile black vllm/ tests/

echo "=== Step 2: Linting ==="
python -m flake8 vllm/ --max-line-length=120 --ignore=E203,W503

echo "=== Step 3: Type checking ==="
python -m mypy vllm/ --ignore-missing-imports --no-strict-optional

echo "=== Step 4: Quick unit tests ==="
python -m pytest tests/core/ -x -q --timeout=60

echo "=== All checks passed! Safe to push. ==="
'''

# Script for running specific test categories
run_tests = '''
#!/bin/bash
# run_tests.sh - Run tests by category

case "$1" in
  unit)
    pytest tests/core/ tests/engine/ -v --timeout=120
    ;;
  model)
    pytest tests/models/ -v --timeout=600
    ;;
  kernel)
    pytest tests/kernels/ -v --timeout=300
    ;;
  api)
    pytest tests/entrypoints/ -v --timeout=300
    ;;
  all)
    pytest tests/ -v --timeout=600
    ;;
  *)
    echo "Usage: $0 {unit|model|kernel|api|all}"
    exit 1
    ;;
esac
'''

print("Development Helper Scripts")
print("=" * 60)
print("\n1. Rebuild Kernels Only:")
print(rebuild_kernels)
print("\n2. Pre-Push Validation:")
print(pre_push_check)
print("\n3. Run Tests by Category:")
print(run_tests)

In [ ]:
# Demo: Git workflow for vLLM/SGLang development

git_workflow = """
Git Workflow for Contributing to vLLM/SGLang
=============================================

1. Fork the repository on GitHub
   - Go to https://github.com/vllm-project/vllm
   - Click "Fork"

2. Clone your fork:
   git clone https://github.com/YOUR_USERNAME/vllm.git
   cd vllm
   
3. Add upstream remote:
   git remote add upstream https://github.com/vllm-project/vllm.git
   
4. Create a feature branch:
   git checkout -b feat/my-feature main
   
5. Make changes, commit:
   git add -A
   git commit -m "feat: add my feature"
   
6. Keep up with upstream:
   git fetch upstream
   git rebase upstream/main
   
7. Push and create PR:
   git push origin feat/my-feature
   # Then create PR on GitHub
   
Commit Message Convention:
  feat: add new feature
  fix: fix a bug
  docs: update documentation
  refactor: code refactoring
  test: add or update tests
  perf: performance improvement
  ci: CI/CD changes
"""

print(git_workflow)

---
## Hands-On Assignments

### Assignment 1: Build vLLM from Source and Run Tests

**Objective:** Successfully build vLLM from source and verify it works.

**Tasks:**
1. Clone the vLLM repository
2. Create a virtual environment
3. Install dependencies and build
4. Run a subset of the unit tests
5. Start the API server with a small model

In [ ]:
# Assignment 1: Starter Code
# Complete the build verification script below

import subprocess
import sys
import os

class VLLMBuildVerifier:
    """Verify a vLLM build from source."""
    
    def __init__(self, vllm_dir="/path/to/vllm"):
        self.vllm_dir = vllm_dir
        self.results = {}
    
    def verify_clone(self):
        """Task 1: Verify the repo is cloned correctly."""
        # TODO: Check that self.vllm_dir exists
        # TODO: Check that it is a git repository
        # TODO: Print the current branch and latest commit
        pass
    
    def verify_environment(self):
        """Task 2: Verify the Python environment is set up."""
        # TODO: Check Python version >= 3.9
        # TODO: Check that we're in a virtual environment
        # TODO: Check that torch is installed with CUDA support
        pass
    
    def verify_build(self):
        """Task 3: Verify vLLM is built and importable."""
        # TODO: Try importing vllm
        # TODO: Print vllm.__version__
        # TODO: Check that CUDA extensions are loaded
        pass
    
    def run_quick_tests(self):
        """Task 4: Run a quick subset of unit tests."""
        # TODO: Run `pytest tests/core/test_block_manager.py -x -q`
        # TODO: Capture and report the results
        pass
    
    def verify_server_starts(self):
        """Task 5: Verify the API server can start."""
        # TODO: Start the server in a subprocess with a small model
        # TODO: Wait for it to be ready (check /health endpoint)
        # TODO: Send a test request
        # TODO: Shut down the server
        pass
    
    def run_all(self):
        """Run all verifications."""
        steps = [
            ("Clone", self.verify_clone),
            ("Environment", self.verify_environment),
            ("Build", self.verify_build),
            ("Tests", self.run_quick_tests),
            ("Server", self.verify_server_starts),
        ]
        for name, func in steps:
            print(f"\n--- Verifying: {name} ---")
            try:
                func()
                self.results[name] = "PASS"
            except Exception as e:
                self.results[name] = f"FAIL: {e}"
        
        print("\n" + "=" * 40)
        print("Summary:")
        for name, result in self.results.items():
            print(f"  {name}: {result}")

# Uncomment and set the correct path to run:
# verifier = VLLMBuildVerifier("/path/to/vllm")
# verifier.run_all()
print("Complete the VLLMBuildVerifier class and run all checks.")

### Assignment 2: Build SGLang from Source

**Objective:** Build SGLang from source and verify the runtime starts correctly.

**Tasks:**
1. Clone SGLang and install in dev mode
2. Verify all components are importable
3. Start the SGLang runtime with a test model
4. Compare the build process with vLLM

In [ ]:
# Assignment 2: Starter Code

class SGLangBuildVerifier:
    """Verify an SGLang build from source."""
    
    def __init__(self, sglang_dir="/path/to/sglang"):
        self.sglang_dir = sglang_dir
    
    def verify_installation(self):
        """Verify SGLang is installed correctly."""
        # TODO: Import sglang
        # TODO: Import sglang.srt (runtime)
        # TODO: Import sglang.lang (language frontend)
        # TODO: Print versions
        pass
    
    def verify_runtime_dependencies(self):
        """Verify all runtime dependencies."""
        deps = [
            'torch', 'transformers', 'tokenizers',
            'fastapi', 'uvicorn', 'triton'
        ]
        # TODO: Check each dependency is importable
        # TODO: Print version of each
        for dep in deps:
            pass  # Implement check
    
    def compare_with_vllm(self):
        """Document differences between vLLM and SGLang build processes."""
        comparison = {
            'Build System': {
                'vLLM': 'setuptools + CMake for CUDA kernels',
                'SGLang': 'setuptools with optional CUDA components'
            },
            'CUDA Kernels': {
                'vLLM': 'Extensive custom kernels in csrc/',
                'SGLang': 'Uses Triton + some custom kernels'
            },
            'Install Modes': {
                'vLLM': 'pip install -e . (single package)',
                'SGLang': 'pip install -e "python[all]" (extras)'
            }
        }
        # TODO: Print formatted comparison
        # TODO: Add your own observations
        pass

# Uncomment and set correct path:
# verifier = SGLangBuildVerifier("/path/to/sglang")
# verifier.verify_installation()
# verifier.verify_runtime_dependencies()
# verifier.compare_with_vllm()
print("Complete the SGLangBuildVerifier class and compare both projects.")

### Assignment 3: Set Up a Complete Dev Environment with Hot Reload

**Objective:** Create a development setup that allows rapid iteration.

**Tasks:**
1. Set up editable install with `pip install -e .`
2. Configure auto-reload for the API server
3. Write a Makefile for common dev tasks
4. Create a `.envrc` file for direnv integration
5. Set up pre-commit hooks

In [ ]:
# Assignment 3: Starter Code

def generate_makefile():
    """Generate a Makefile for common dev tasks."""
    # TODO: Complete the Makefile content
    makefile_content = '''
.PHONY: install dev test lint format clean serve

# Install in development mode
install:
\t# TODO: pip install command for dev mode

# Install development dependencies
dev:
\t# TODO: install dev dependencies (pytest, black, isort, etc.)

# Run tests
test:
\t# TODO: pytest command

test-unit:
\t# TODO: run only unit tests

test-model:
\t# TODO: run model correctness tests

# Lint and format
lint:
\t# TODO: run linting checks

format:
\t# TODO: auto-format code

# Clean build artifacts
clean:
\t# TODO: remove build directories

# Start dev server
serve:
\t# TODO: start the API server with a small model

# Rebuild CUDA kernels only
kernels:
\t# TODO: rebuild kernels only
'''
    return makefile_content

def generate_envrc():
    """Generate a .envrc for direnv integration."""
    # TODO: Add environment variables
    envrc = '''
# .envrc - direnv configuration
# Run: direnv allow . to enable

# TODO: Activate virtual environment
# TODO: Set CUDA_HOME
# TODO: Set VLLM_LOGGING_LEVEL for development
# TODO: Set MAX_JOBS for builds
'''
    return envrc

print("Task: Complete the Makefile and .envrc templates")
print("\n--- Makefile template ---")
print(generate_makefile())
print("\n--- .envrc template ---")
print(generate_envrc())
print("\nFill in the TODO sections to create a working development setup.")

In [ ]:
# Assignment 3 continued: Pre-commit hook verification

def verify_precommit_setup(repo_dir):
    """Verify pre-commit hooks are properly configured."""
    import os
    
    checks = {
        "pre-commit installed": False,
        "config file exists": False,
        "hooks installed in git": False,
        "hooks run successfully": False,
    }
    
    # TODO: Check if pre-commit is installed (pip)
    # TODO: Check if .pre-commit-config.yaml exists
    # TODO: Check if .git/hooks/pre-commit exists
    # TODO: Run pre-commit on a test file
    
    print("Pre-commit Setup Verification:")
    for check, status in checks.items():
        symbol = "[OK]" if status else "[TODO]"
        print(f"  {symbol} {check}")

# Uncomment to run:
# verify_precommit_setup("/path/to/vllm")
print("Complete the verify_precommit_setup function and run it on your clone.")

---
## Summary

In this notebook, we covered:

1. **Prerequisites** - Checking system requirements for CUDA, Python, and build tools
2. **Python environments** - Using conda/venv for isolated development
3. **Building vLLM** - Step-by-step source build with environment variables
4. **Building SGLang** - Source build and project structure
5. **IDE setup** - VSCode configuration with debugging support
6. **Pre-commit hooks** - Automated code quality checks
7. **Development server** - Running vLLM/SGLang in debug mode
8. **Troubleshooting** - Common build errors and solutions

**Next:** Chapter 11.2 - Testing Framework